In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import pathlib
import numpy as np

DATA_DIR = pathlib.Path.home() / "taekwondo-scorer/data/태극 1장 데이터"

# JSON 파일 하나 열어보기
sample_json = list(DATA_DIR.glob("*/*/*.json"))[0]
print("샘플 파일:", sample_json)

with open(sample_json, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"\n총 프레임 수: {len(data['labelingInfo'])}")
print(f"\n관절 목록:")
joints = list(data['labelingInfo'][0]['pose']['location'].keys())
for i, j in enumerate(joints):
    print(f"  {i}: {j}")

In [ ]:
import numpy as np

# ← 골반 → 엉덩이로 수정!
JOINT_MAP = {
    "코": 0,
    "왼쪽 눈": 1,
    "오른쪽 눈": 2,
    "왼쪽 귀": 3,
    "오른쪽 귀": 4,
    "왼쪽 어깨": 5,
    "오른쪽 어깨": 6,
    "왼쪽 팔꿈치": 7,
    "오른쪽 팔꿈치": 8,
    "왼쪽 손목": 9,
    "오른쪽 손목": 10,
    "왼쪽 엉덩이": 11,   # 골반 → 엉덩이로 수정
    "오른쪽 엉덩이": 12,  # 골반 → 엉덩이로 수정
    "왼쪽 무릎": 13,
    "오른쪽 무릎": 14,
    "왼쪽 발목": 15,
    "오른쪽 발목": 16,
}

def parse_sequence(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    frames = []
    for frame_info in data['labelingInfo']:
        if 'pose' not in frame_info:
            continue
        if 'location' not in frame_info['pose']:
            continue
            
        location = frame_info['pose']['location']
        keypoints = np.zeros((17, 2))
        
        for joint_name, idx in JOINT_MAP.items():
            if joint_name in location:
                x = float(location[joint_name]['x'])
                y = float(location[joint_name]['y'])
                keypoints[idx] = [x, y]
        
        frames.append(keypoints)
    
    if len(frames) == 0:
        return None
    
    return np.array(frames)

# 테스트
seq = parse_sequence(sample_json)
print(f"시퀀스 shape: {seq.shape}")
print(f"T={seq.shape[0]}프레임, 관절={seq.shape[1]}개")

In [ ]:
import matplotlib.pyplot as plt

SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]

def visualize_frame(seq, frame_idx=0, title=""):
    fig, ax = plt.subplots(figsize=(5, 8))
    kps = seq[frame_idx]
    
    for (i, j) in SKELETON:
        if kps[i].sum() > 0 and kps[j].sum() > 0:
            ax.plot([kps[i,0], kps[j,0]],
                   [kps[i,1], kps[j,1]], 'b-', linewidth=2)
    
    ax.scatter(kps[:,0], kps[:,1], c='red', s=80, zorder=5)
    ax.set_title(f"Frame {frame_idx} {title}")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

# 이 시퀀스는 1프레임이라 0번만 가능
visualize_frame(seq, frame_idx=0, title="(자세 확인)")
print("사람 형태로 보이면 성공!")

In [ ]:
import pathlib

DATA_DIR = pathlib.Path.home() / "taekwondo-scorer/data/태극 1장 데이터"
PROCESSED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed"
PROCESSED_DIR.mkdir(exist_ok=True)

# 전체 JSON 파일 찾기
all_json = list(DATA_DIR.glob("*/*/*.json"))
print(f"전체 시퀀스 수: {len(all_json)}개")

# 동작별로 몇 개나 있는지 확인
from collections import Counter
action_counts = Counter()
for j in all_json:
    action_name = j.parent.parent.name  # 동작명 폴더
    action_counts[action_name] += 1

print("\n동작별 시퀀스 수:")
for action, count in sorted(action_counts.items()):
    print(f"  {action}: {count}개")

In [ ]:
import json
import numpy as np
import pathlib

DATA_DIR = pathlib.Path.home() / "taekwondo-scorer/data/태극 1장 데이터"
PROCESSED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed"
PROCESSED_DIR.mkdir(exist_ok=True)

JOINT_MAP = {
    "코": 0, "왼쪽 눈": 1, "오른쪽 눈": 2,
    "왼쪽 귀": 3, "오른쪽 귀": 4,
    "왼쪽 어깨": 5, "오른쪽 어깨": 6,
    "왼쪽 팔꿈치": 7, "오른쪽 팔꿈치": 8,
    "왼쪽 손목": 9, "오른쪽 손목": 10,
    "왼쪽 엉덩이": 11, "오른쪽 엉덩이": 12,
    "왼쪽 무릎": 13, "오른쪽 무릎": 14,
    "왼쪽 발목": 15, "오른쪽 발목": 16,
}

def parse_sequence(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    frames = []
    for frame_info in data['labelingInfo']:
        if 'pose' not in frame_info:
            continue
        if 'location' not in frame_info['pose']:
            continue
        location = frame_info['pose']['location']
        keypoints = np.zeros((17, 2))
        for joint_name, idx in JOINT_MAP.items():
            if joint_name in location:
                keypoints[idx] = [float(location[joint_name]['x']),
                                   float(location[joint_name]['y'])]
        frames.append(keypoints)
    return np.array(frames) if frames else None

# 전체 처리
all_json = list(DATA_DIR.glob("*/*/*.json"))
total = len(all_json)
success, fail, skip = 0, 0, 0

print(f"총 {total}개 처리 시작...")

for i, json_path in enumerate(all_json):
    try:
        action_name = json_path.parent.parent.name
        person_name = json_path.parent.name

        save_dir = PROCESSED_DIR / action_name / person_name
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / (json_path.stem + ".npy")

        if save_path.exists():
            skip += 1
        else:
            seq = parse_sequence(json_path)
            if seq is None or len(seq) == 0:
                fail += 1
            else:
                np.save(save_path, seq)
                success += 1

    except Exception as e:
        fail += 1

    # 5000개마다 진행상황 출력
    if (i + 1) % 5000 == 0:
        print(f"  [{i+1}/{total}] 성공:{success} 실패:{fail} 건너뜀:{skip}")

print(f"\n완료!")
print(f"  성공: {success}개")
print(f"  건너뜀: {skip}개")
print(f"  실패: {fail}개")

In [ ]:
import numpy as np
import pathlib

PROCESSED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed"
NORMALIZED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed/normalized"
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

def normalize_sequence(seq):
    """
    (T, 17, 2) 배열을 정규화
    1. 위치 정규화: 골반 중점을 (0,0)으로
    2. 크기 정규화: 어깨-골반 거리로 나눔
    """
    normalized = seq.copy().astype(float)
    T = len(seq)
    
    for t in range(T):
        kps = normalized[t]  # (17, 2)
        
        # 1. 위치 정규화: 골반 중점(엉덩이 11,12)을 (0,0)으로
        left_hip  = kps[11]   # 왼쪽 엉덩이
        right_hip = kps[12]   # 오른쪽 엉덩이
        
        if left_hip.sum() == 0 or right_hip.sum() == 0:
            continue
            
        hip_center = (left_hip + right_hip) / 2
        kps -= hip_center
        
        # 2. 크기 정규화: 어깨 중점 - 골반 중점 거리로 나눔
        left_shoulder  = kps[5]
        right_shoulder = kps[6]
        shoulder_center = (left_shoulder + right_shoulder) / 2
        
        scale = np.linalg.norm(shoulder_center)  # 골반 기준 어깨까지 거리
        if scale > 0:
            kps /= scale
        
        normalized[t] = kps
    
    return normalized

def joint_angle(a, b, c):
    """b를 꼭짓점으로 하는 a-b-c 각도 계산"""
    ba = a - b
    bc = c - b
    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)
    if norm_ba == 0 or norm_bc == 0:
        return 0.0
    cos = np.dot(ba, bc) / (norm_ba * norm_bc)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def extract_angles(seq):
    """
    (T, 17, 2) → (T, 8) 관절 각도 시퀀스
    채점에 중요한 8개 관절 각도 추출
    """
    T = len(seq)
    angles = np.zeros((T, 8))
    
    for t in range(T):
        kps = seq[t]
        # 0:코 1:왼눈 2:오른눈 3:왼귀 4:오른귀
        # 5:왼어깨 6:오른어깨 7:왼팔꿈치 8:오른팔꿈치
        # 9:왼손목 10:오른손목 11:왼엉덩이 12:오른엉덩이
        # 13:왼무릎 14:오른무릎 15:왼발목 16:오른발목
        
        angles[t, 0] = joint_angle(kps[5],  kps[7],  kps[9])   # 왼팔꿈치
        angles[t, 1] = joint_angle(kps[6],  kps[8],  kps[10])  # 오른팔꿈치
        angles[t, 2] = joint_angle(kps[7],  kps[5],  kps[11])  # 왼어깨
        angles[t, 3] = joint_angle(kps[8],  kps[6],  kps[12])  # 오른어깨
        angles[t, 4] = joint_angle(kps[11], kps[13], kps[15])  # 왼무릎
        angles[t, 5] = joint_angle(kps[12], kps[14], kps[16])  # 오른무릎
        angles[t, 6] = joint_angle(kps[5],  kps[11], kps[13])  # 왼엉덩이
        angles[t, 7] = joint_angle(kps[6],  kps[12], kps[14])  # 오른엉덩이
    
    return angles

# 샘플 하나로 테스트
sample_npy = list(PROCESSED_DIR.glob("*/*/*.npy"))[0]
print("테스트 파일:", sample_npy.name)

seq = np.load(sample_npy)
print(f"원본 shape: {seq.shape}")

# 정규화
norm_seq = normalize_sequence(seq)
print(f"정규화 후 골반 중점: {((norm_seq[:,11] + norm_seq[:,12]) / 2).mean(axis=0)}")  # 0에 가까워야 함

# 관절 각도 추출
angles = extract_angles(norm_seq)
print(f"관절 각도 shape: {angles.shape}")
print(f"첫 프레임 각도: {angles[0].round(1)}")
print("\n각도 이름: 왼팔꿈치, 오른팔꿈치, 왼어깨, 오른어깨, 왼무릎, 오른무릎, 왼엉덩이, 오른엉덩이")

In [ ]:
import numpy as np
import pathlib

PROCESSED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed"
NORMALIZED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed/normalized"
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

all_npy = [p for p in PROCESSED_DIR.glob("*/*/*.npy")]
total = len(all_npy)
success, fail, skip = 0, 0, 0

print(f"총 {total}개 정규화 시작...")

for i, npy_path in enumerate(all_npy):
    try:
        action_name = npy_path.parent.parent.name
        person_name = npy_path.parent.name

        save_dir = NORMALIZED_DIR / action_name / person_name
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / npy_path.name

        if save_path.exists():
            skip += 1
            continue

        seq = np.load(npy_path)
        norm_seq = normalize_sequence(seq)
        angles = extract_angles(norm_seq)

        # 각도 시퀀스 저장 (T, 8)
        np.save(save_path, angles)
        success += 1

    except Exception as e:
        fail += 1

    if (i + 1) % 5000 == 0:
        print(f"  [{i+1}/{total}] 성공:{success} 실패:{fail} 건너뜀:{skip}")

print(f"\n완료!")
print(f"  성공: {success}개")
print(f"  건너뜀: {skip}개")
print(f"  실패: {fail}개")

In [ ]:
import numpy as np
import pathlib
from scipy.interpolate import interp1d
from collections import defaultdict

NORMALIZED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed/normalized"
TEMPLATES_DIR = pathlib.Path.home() / "taekwondo-scorer/templates"
TEMPLATES_DIR.mkdir(exist_ok=True)

TARGET_LEN = 60  # 모든 시퀀스를 60프레임으로 통일

def resample_sequence(seq, target_len=60):
    """시퀀스를 target_len 프레임으로 보간"""
    T, J = seq.shape
    if T == 1:
        # 1프레임이면 복제
        return np.tile(seq, (target_len, 1))
    f = interp1d(np.linspace(0, 1, T), seq, axis=0)
    return f(np.linspace(0, 1, target_len))

# 동작별로 시퀀스 모으기
print("동작별 시퀀스 로딩 중...")
action_sequences = defaultdict(list)

all_npy = list(NORMALIZED_DIR.glob("*/*/*.npy"))
print(f"전체 파일: {len(all_npy)}개")

for npy_path in all_npy:
    action_name = npy_path.parent.parent.name
    try:
        seq = np.load(npy_path)  # (T, 8)
        if seq is None or len(seq) == 0:
            continue
        resampled = resample_sequence(seq, TARGET_LEN)  # (60, 8)
        action_sequences[action_name].append(resampled)
    except:
        continue

print(f"\n동작별 시퀀스 수:")
for action, seqs in sorted(action_sequences.items()):
    print(f"  {action}: {len(seqs)}개")

In [ ]:
# 템플릿 평균 계산 및 저장
print("템플릿 생성 중...")

for action, seqs in action_sequences.items():
    stack = np.stack(seqs, axis=0)      # (N, 60, 8)
    template = np.mean(stack, axis=0)   # (60, 8)
    
    save_path = TEMPLATES_DIR / f"{action}.npy"
    np.save(save_path, template)
    print(f"  ✅ {action}: {len(seqs)}개 평균 → {template.shape} 저장완료")

print(f"\n총 {len(action_sequences)}개 템플릿 생성 완료!")
print(f"저장 위치: {TEMPLATES_DIR}")

# 저장된 템플릿 확인
print("\n저장된 템플릿 파일:")
for t in sorted(TEMPLATES_DIR.glob("*.npy")):
    template = np.load(t)
    print(f"  {t.name}: {template.shape}")

In [ ]:
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import numpy as np
import pathlib

TEMPLATES_DIR = pathlib.Path.home() / "taekwondo-scorer/templates"
NORMALIZED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed/normalized"

# 템플릿 전체 로딩
templates = {}
for t in sorted(TEMPLATES_DIR.glob("*.npy")):
    templates[t.stem] = np.load(t)
    print(f"  로딩: {t.stem} {templates[t.stem].shape}")

print(f"\n총 {len(templates)}개 템플릿 로딩 완료!")

def resample_sequence(seq, target_len=60):
    from scipy.interpolate import interp1d
    T, J = seq.shape
    if T == 1:
        return np.tile(seq, (target_len, 1))
    f = interp1d(np.linspace(0, 1, T), seq, axis=0)
    return f(np.linspace(0, 1, target_len))

def score_sequence(seq, action_name):
    """
    seq: (T, 8) 관절 각도 시퀀스
    action_name: 동작 이름
    반환: 0~100 점수, 관절별 오차
    """
    if action_name not in templates:
        return None, None
    
    template = templates[action_name]  # (60, 8)
    
    # 60프레임으로 보간
    resampled = resample_sequence(seq, 60)  # (60, 8)
    
    # DTW 거리 계산
    distance, path = fastdtw(resampled, template, dist=euclidean)
    
    # 관절별 평균 오차 계산 (마지막 프레임 기준)
    joint_errors = np.abs(resampled[-1] - template[-1])
    
    # 점수 변환 (거리가 작을수록 높은 점수)
    # 거리 0 → 100점, 거리 클수록 낮아짐
    score = max(0, 100 - distance * 2)
    score = min(100, score)
    
    return round(score, 1), joint_errors

# 테스트: 템플릿 자체를 입력하면 100점에 가까워야 함
print("\n=== 템플릿 자체 채점 테스트 ===")
for action_name, template in list(templates.items())[:3]:
    score, errors = score_sequence(template, action_name)
    print(f"  {action_name}: {score}점")

# 테스트: 실제 데이터 하나 채점
print("\n=== 실제 데이터 채점 테스트 ===")
sample_npy = list(NORMALIZED_DIR.glob("앞서고 지르기/*/*.npy"))[0]
seq = np.load(sample_npy)
score, errors = score_sequence(seq, "앞서고 지르기")
print(f"  앞서고 지르기 샘플: {score}점")

joint_names = ["왼팔꿈치", "오른팔꿈치", "왼어깨", "오른어깨", "왼무릎", "오른무릎", "왼엉덩이", "오른엉덩이"]
print(f"\n관절별 오차:")
for name, err in zip(joint_names, errors):
    print(f"  {name}: {err:.1f}도")

In [ ]:
# DTW 거리 실제값 확인
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
from scipy.interpolate import interp1d

def resample_sequence(seq, target_len=60):
    T, J = seq.shape
    if T == 1:
        return np.tile(seq, (target_len, 1))
    f = interp1d(np.linspace(0, 1, T), seq, axis=0)
    return f(np.linspace(0, 1, target_len))

# 여러 샘플의 DTW 거리 분포 확인
action_name = "앞서고 지르기"
template = templates[action_name]

distances = []
all_samples = list(NORMALIZED_DIR.glob(f"{action_name}/*/*.npy"))[:100]  # 100개 샘플

for npy_path in all_samples:
    try:
        seq = np.load(npy_path)
        resampled = resample_sequence(seq, 60)
        dist, _ = fastdtw(resampled, template, dist=euclidean)
        distances.append(dist)
    except:
        continue

distances = np.array(distances)
print(f"DTW 거리 분포 (앞서고 지르기, {len(distances)}개 샘플):")
print(f"  최솟값: {distances.min():.1f}")
print(f"  평균값: {distances.mean():.1f}")
print(f"  최댓값: {distances.max():.1f}")
print(f"  중앙값: {np.median(distances):.1f}")
print(f"  표준편차: {distances.std():.1f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.hist(distances, bins=30, color='steelblue', edgecolor='white')
plt.xlabel("DTW 거리")
plt.ylabel("빈도")
plt.title(f"{action_name} DTW 거리 분포")
plt.tight_layout()
plt.show()

In [ ]:
# 동작별 거리 분포 기반으로 점수 매핑 테이블 만들기
print("전체 동작별 거리 분포 계산 중...")

distance_stats = {}

for action_name in templates.keys():
    template = templates[action_name]
    all_samples = list(NORMALIZED_DIR.glob(f"{action_name}/*/*.npy"))[:200]
    
    distances = []
    for npy_path in all_samples:
        try:
            seq = np.load(npy_path)
            resampled = resample_sequence(seq, 60)
            dist, _ = fastdtw(resampled, template, dist=euclidean)
            distances.append(dist)
        except:
            continue
    
    if distances:
        distances = np.array(distances)
        distance_stats[action_name] = {
            'min': distances.min(),
            'p10': np.percentile(distances, 10),   # 상위 10% → 90점대
            'p25': np.percentile(distances, 25),   # 상위 25% → 80점대
            'p50': np.percentile(distances, 50),   # 중간 → 60점대
            'p75': np.percentile(distances, 75),   # 하위 25% → 40점대
            'p90': np.percentile(distances, 90),   # 하위 10% → 20점대
            'max': distances.max(),
        }
        print(f"  {action_name}: min={distances.min():.0f}, p50={np.percentile(distances,50):.0f}, max={distances.max():.0f}")

# 개선된 채점 함수
def score_sequence_v2(seq, action_name):
    if action_name not in templates:
        return None, None
    
    template = templates[action_name]
    resampled = resample_sequence(seq, 60)
    distance, _ = fastdtw(resampled, template, dist=euclidean)
    
    stats = distance_stats[action_name]
    
    # 백분위 기반 점수 매핑
    # 상위 10%(p10 이하) → 85~100점
    # 중간(p50)         → 55~60점
    # 하위 10%(p90 이상) → 20점 이하
    if distance <= stats['min']:
        score = 100
    elif distance <= stats['p10']:
        score = 85 + 15 * (1 - (distance - stats['min']) / (stats['p10'] - stats['min']))
    elif distance <= stats['p25']:
        score = 70 + 15 * (1 - (distance - stats['p10']) / (stats['p25'] - stats['p10']))
    elif distance <= stats['p50']:
        score = 55 + 15 * (1 - (distance - stats['p25']) / (stats['p50'] - stats['p25']))
    elif distance <= stats['p75']:
        score = 40 + 15 * (1 - (distance - stats['p50']) / (stats['p75'] - stats['p50']))
    elif distance <= stats['p90']:
        score = 20 + 20 * (1 - (distance - stats['p75']) / (stats['p90'] - stats['p75']))
    else:
        score = max(0, 20 * (1 - (distance - stats['p90']) / (stats['max'] - stats['p90'] + 1)))
    
    # 관절별 오차
    joint_errors = np.abs(resampled[-1] - template[-1])
    
    return round(score, 1), joint_errors, distance

# 테스트
print("\n=== 채점 테스트 ===")
joint_names = ["왼팔꿈치", "오른팔꿈치", "왼어깨", "오른어깨", "왼무릎", "오른무릎", "왼엉덩이", "오른엉덩이"]

# 1. 여러 샘플 채점
action_name = "앞서고 지르기"
samples = list(NORMALIZED_DIR.glob(f"{action_name}/*/*.npy"))[:10]

scores = []
for s in samples:
    seq = np.load(s)
    score, errors, dist = score_sequence_v2(seq, action_name)
    scores.append(score)
    print(f"  점수: {score}점  (DTW거리: {dist:.0f})")

print(f"\n평균 점수: {np.mean(scores):.1f}점")
print(f"최고 점수: {max(scores):.1f}점")
print(f"최저 점수: {min(scores):.1f}점")

In [ ]:
import json
import pathlib

# distance_stats를 파일로 저장
STATS_PATH = pathlib.Path.home() / "taekwondo-scorer/distance_stats.json"

# float으로 변환해서 저장
save_stats = {}
for action, stat in distance_stats.items():
    save_stats[action] = {k: float(v) for k, v in stat.items()}

with open(STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(save_stats, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {STATS_PATH}")
print(json.dumps(save_stats, ensure_ascii=False, indent=2)[:500])

In [ ]:
print(distance_stats)

In [ ]:
ㄴ